# Growth: what if we opened a typical lounge here?

**What changed from v3:** an area's call comes from a what-if, not from size and saturation tests. A typical lounge (median lounge reviews and rating) opens at the area's centroid; the Huff split gives the women it serves, how many it takes from our lounges, and the net new. GROW = net new ≥ `NEW_SITE_MIN`, worker housing under 50%, residential name, `rent_ratio` ≥ 1 (no dearer than its emirate's median salon site, or unknown) and `demand_pct` ≥ 0.5 (out-serves at least half its emirate's reference salon sites; UAE-wide under 50 sites).

**The model, as agreed on 2026-10-11** (one model for lounges and growth):
1. **Demand = served women, Huff** (`src/model/served.py`). Each cell's addressable women are split, Huff-style, across every premium salon and every lounge within 20 minutes of it, across emirate borders. Weight = reviews^ALPHA × (rating/4.6)^RATING_ELASTICITY × DECAY[drive-time band], plus an outside option (women who go elsewhere or use no premium salon). Drive-time bands come from the cell's own isochrones, else from straight-line bands fitted on those. Cells our salon search never covered get an imputed stand-in competitor (emirate × density tier).
2. **Cost = shop rent at the site** (`src/features/lounges.py`). Shop rent per cell from three sources (2026-10-11): DLD Ejari contracts (Dubai), ADREC 2025 district rates (Abu Dhabi), shop listings' asking rents (northern emirates), else the emirate median. The cost signal is `rent_ratio` = the emirate's median shop rent at its reference salon sites ÷ the site's rent (> 1 = cheaper than the typical salon site in its emirate), scored on a frozen scale: `RENT_RATIO_P10` = 0.6176 → 0, `RENT_RATIO_P90` = 1.4290 → 1, so a ratio of 1 scores 0.47 (`RENT_RATIO_WHY`). Rent only: served women are the demand signal and aren't counted twice. `site_cost_k` (rent × 150 m² + an assumed fit-out over 5 years) is shown, not scored.
3. **Lounge scorecard** = site demand + site rent, both judged **within the emirate** against a frozen reference of 1,123 real premium competitor salon sites (no imputed, no Bedashing; `data/seed/v3/site_reference.csv`, `SITE_REFERENCE_WHY`): Dubai 582, Abu Dhabi 304, Sharjah 143, Ajman 38, Ras Al Khaimah 33, Fujairah 18, Umm Al Quwain 5. Never against our own lounges, so opening or closing one moves nothing. Site demand (`site_served`) = the women a typical lounge (median reviews and rating) would serve at this lounge's site, with this lounge closed and its siblings and rivals open; it is scored as `demand_pct`, the share of the emirate's reference sites it out-serves (UAE-wide for the four emirates under 50 sites; `demand_ref` says which). `demand_pct_uae` and `rent_pct_uae` give the UAE-wide percentiles as context, not scored. The lounge's own served women are context too: `performance_vs_site` = served ÷ site-served ("The model estimates this lounge is performing 2.7x compared to its site"). Cannibalisation (the share of a lounge's women its siblings would absorb if it closed) is shown, not scored.
4. **Growth** = what-if: open a typical lounge (median lounge reviews and rating) at each area's centroid, and count the women it serves, how many it takes from our lounges, and the net new. GROW needs net new ≥ 6,000, worker housing under 50%, `rent_ratio` ≥ 1 (unknown passes) and `demand_pct` ≥ 0.5, all against the same salon-site reference.

**Every parameter here is a first cut**, not calibrated. This notebook shows what they produce so we can decide together.

In [1]:
import sys, pathlib, statistics
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_rows", 120)
from src.baseline import run
from src.data_v3 import load_v3
from src.config import REPO_ROOT, load_baseline_assumptions
from src.model import served as S, scorecard, growth
from src.features import lounges as L
r = run(); v3 = load_v3(); a = load_baseline_assumptions(REPO_ROOT / "data/scenarios/baseline.yaml")
old = pd.read_csv(REPO_ROOT / "data/scenarios/baseline_v3_labels.csv")
old_action = dict(zip(old.id, old.action))
F = pd.DataFrame([x.model_dump() for x in r.features]).set_index("branch_id")
D = {d.branch_id: d for d in r.decisions}
AR = pd.DataFrame([x.model_dump(exclude={"cell_ids"}) for x in r.areas]).set_index("area_id")
AR["call"] = AR.index.map({d.area_id: d.action for d in r.area_decisions})

## 1. The typical new lounge and the GROW line

In [2]:
scored = F[~F.not_scored]
rev, rat = statistics.median(scored.review_count), statistics.median(scored.rating.dropna())
print("typical new lounge: reviews", rev, "| rating", rat)
print("NEW_SITE_MIN", growth.NEW_SITE_MIN, "|", growth.NEW_SITE_MIN_WHY)
print("calls:", AR.call.value_counts().to_dict())

typical new lounge: reviews 757 | rating 4.6
NEW_SITE_MIN 6000 | Quick first cut (2026-10-11): a new lounge here should add at least about what our median lounge serves today (~6k women, Huff), net of what it takes from our own lounges.
calls: {'SKIP': 464, 'WATCH': 65, 'GROW': 32}


## 2. Per area: served, taken, net new, demand percentile, rent ratio, data coverage; which GROW test each WATCH fails

In [3]:
cols = ["emirate", "women", "new_site_served", "new_site_taken", "new_site_net", "demand_ref", "demand_pct", "demand_pct_uae", "rent_ratio", "rent_pct_uae",
        "worker_share", "data_coverage", "nearest_lounge_id", "nearest_lounge_km", "call"]
t = AR[AR.call != "SKIP"][cols].assign(v3=lambda x: x.index.map(lambda i: old_action.get(i, "(new area)")))
def fails(x):
    if x.call != "WATCH": return ""
    return ", ".join(n for n, bad in (("net new", x.new_site_net < growth.NEW_SITE_MIN), ("worker", x.worker_share >= growth.WORKER_CAP),
                                       ("rent", x.rent_ratio == x.rent_ratio and x.rent_ratio < growth.RENT_RATIO_MIN),
                                       ("demand", x.demand_pct == x.demand_pct and x.demand_pct < growth.DEMAND_PCT_MIN)) if bad)
t["fails"] = t.apply(fails, axis=1)
t = t.sort_values(["call", "new_site_net"], ascending=[True, False])
print("WATCH by failed test(s):", t[t.call == "WATCH"].fails.value_counts().to_dict())
t.round(2)

WATCH by failed test(s): {'net new': 29, 'rent': 17, 'net new, rent': 16, 'net new, rent, demand': 2, 'net new, demand': 1}


,emirate,women,new_site_served,new_site_taken,new_site_net,demand_ref,demand_pct,demand_pct_uae,rent_ratio,rent_pct_uae,worker_share,data_coverage,nearest_lounge_id,nearest_lounge_km,call,v3,fails
area_id,,,,,,,,,,,,,,,,,
al-dhaid-sharjah,Sharjah,73067.03,57931.72,0.00,57931.72,Sharjah,1.00,1.00,1.00,0.20,0.02,0.82,al-jada,43.78,GROW,GROW,
khor-fakkan-sharjah,Sharjah,31162.57,23432.62,8.25,23424.37,Sharjah,0.97,1.00,1.00,0.20,0.01,0.94,al-taif-mall,26.50,GROW,GROW,
al-madam-sharjah,Sharjah,38980.18,19727.77,0.00,19727.77,Sharjah,0.94,0.99,1.00,0.20,0.01,0.37,mirdif-35,46.72,GROW,WATCH,
milehah-sharjah,Sharjah,13716.41,18518.19,9.45,18508.74,Sharjah,0.94,0.99,1.00,0.20,0.01,0.52,al-taif-mall,43.26,GROW,WATCH,
hamidiya-ajman,Ajman,24639.37,13535.53,149.07,13386.46,UAE,0.98,0.98,1.00,0.07,0.00,1.00,al-jada,10.59,GROW,WATCH,
um-ghafah-abu-dhabi,Abu Dhabi,7694.15,13207.29,0.00,13207.29,Abu Dhabi,0.99,0.98,4.59,0.00,0.00,0.86,al-ain,29.91,GROW,WATCH,
sharjah-sharjah,Sharjah,281946.49,13176.39,368.01,12808.38,Sharjah,0.90,0.98,1.00,0.20,0.12,0.99,al-jada,2.48,GROW,GROW,
mazyad-abu-dhabi,Abu Dhabi,6521.85,12714.33,2.73,12711.60,Abu Dhabi,0.99,0.98,1.34,0.14,0.00,0.93,al-ain,27.92,GROW,WATCH,
al-ramlah-sharjah,Sharjah,12735.18,12600.11,26.92,12573.19,Sharjah,0.90,0.98,1.00,0.20,0.11,1.00,al-jada,17.08,GROW,WATCH,


## 3. v3 → new

In [4]:
oa = old[old.kind == "area"].set_index("id").action
both = pd.DataFrame({"v3": oa, "new": AR.call}).fillna("(absent)")
print(pd.crosstab(both.v3, both.new))
print("\nv3 GROW now:", both[both.v3 == "GROW"].new.to_dict())

new       GROW  SKIP  WATCH
v3                         
(absent)     4   115     29
GROW         5     0      0
SKIP         9   345     18
WATCH       14     4     18

v3 GROW now: {'al-dhaid-sharjah': 'GROW', 'al-jerf-ajman': 'GROW', 'kalba-sharjah': 'GROW', 'khor-fakkan-sharjah': 'GROW', 'sharjah-sharjah': 'GROW'}


## 4. Inside our footprint or remote?

In [5]:
g = t[t.call == "GROW"]
print("GROW within 15 km of a lounge:", int((g.nearest_lounge_km <= 15).sum()), "of", len(g), list(g.index[g.nearest_lounge_km <= 15]))
print("GROW 30+ km from any lounge:", int((g.nearest_lounge_km >= 30).sum()), "of", len(g))
print("GROW by emirate:", g.emirate.value_counts().to_dict())
print("GROW with data_coverage < 0.5 (mostly imputed competitors):", int((g.data_coverage < 0.5).sum()), "of", len(g))
print("GROW with a rent ratio:", g.rent_ratio.notna().sum(), "| demand ranked UAE-wide:", int((g.demand_ref == "UAE").sum()))
print("GROW demand_pct (own reference) min/median:", round(g.demand_pct.min(), 2), round(g.demand_pct.median(), 2),
      "| demand_pct_uae min/median:", round(g.demand_pct_uae.min(), 2), round(g.demand_pct_uae.median(), 2))
print("GROW rent_ratio min/median:", round(g.rent_ratio.min(), 2), round(g.rent_ratio.median(), 2))

GROW within 15 km of a lounge: 17 of 32 ['hamidiya-ajman', 'sharjah-sharjah', 'al-jerf-ajman', 'kalba-sharjah', 'muntazy-ajman', 'zakhir-abu-dhabi', 'meshairef-ajman', 'al-maqam-abu-dhabi', 'al-rawdah-al-sharqiyah-abu-dhabi', 'falaj-hazza-abu-dhabi', 'al-fahlain-ras-al-khaimah', 'khatt-ras-al-khaimah', 'al-rahbah-abu-dhabi', 'al-awir-dubai', 'al-khibeesi-abu-dhabi', 'digdaga-ras-al-khaimah', 'al-shahamah-abu-dhabi']
GROW 30+ km from any lounge: 7 of 32
GROW by emirate: {'Abu Dhabi': 13, 'Sharjah': 10, 'Ajman': 4, 'Ras Al Khaimah': 4, 'Dubai': 1}
GROW with data_coverage < 0.5 (mostly imputed competitors): 7 of 32
GROW with a rent ratio: 32 | demand ranked UAE-wide: 8
GROW demand_pct (own reference) min/median: 0.78 0.9 | demand_pct_uae min/median: 0.79 0.93
GROW rent_ratio min/median: 1.0 1.06


## 5. Calibration: a typical new lounge out-pulls thin markets

Compare what a new lounge would serve with what our existing lounges serve today, and with what a typical lounge would serve at their sites (`site_served`, the scorecard's demand signal: like for like with the what-if).

In [6]:
best = scored.served_women.max()
print("our best lounge serves:", round(best), f"({scored.served_women.idxmax()}) | median lounge:", round(scored.served_women.median()))
print("best lounge site (typical lounge there):", round(scored.site_served.max()), f"({scored.site_served.idxmax()}) | median site:", round(scored.site_served.median()))
w_ = AR[AR.new_site_served.notna()]
print("areas where a new lounge would serve more than our best lounge:", int((w_.new_site_served > best).sum()),
      "| of them GROW:", int(((w_.new_site_served > best) & (w_.call == "GROW")).sum()))
top = w_.sort_values("new_site_served", ascending=False).head(10)[["emirate", "new_site_served", "new_site_net", "data_coverage", "nearest_lounge_km", "call"]]
print("GROW areas where the new lounge would serve more women than the area itself holds (it draws on neighbours within 20 min):",
      int((g.new_site_served > g.women).sum()), "of", len(g))
print("GROW areas within 10 km of another GROW area (the what-ifs are run one at a time, so neighbours count the same women):",
      sum(any(0 < L.haversine_km(x.lat, x.lng, y.lat, y.lng) <= 10 for _, y in AR.loc[g.index].iterrows()) for _, x in AR.loc[g.index].iterrows()))
top.assign(x_best=top.new_site_served / best).round(2)

our best lounge serves: 18652 (al-ain) | median lounge: 6046
best lounge site (typical lounge there): 22852 (al-dhafra) | median site: 5388
areas where a new lounge would serve more than our best lounge: 4 | of them GROW: 3
GROW areas where the new lounge would serve more women than the area itself holds (it draws on neighbours within 20 min): 15 of 32
GROW areas within 10 km of another GROW area (the what-ifs are run one at a time, so neighbours count the same women): 19


,emirate,new_site_served,new_site_net,data_coverage,nearest_lounge_km,call,x_best
area_id,,,,,,,
al-dhaid-sharjah,Sharjah,57931.72,57931.72,0.82,43.78,GROW,3.11
manama-ajman,Ajman,35094.21,35094.21,1.00,39.30,WATCH,1.88
khor-fakkan-sharjah,Sharjah,23432.62,23424.37,0.94,26.50,GROW,1.26
al-madam-sharjah,Sharjah,19727.77,19727.77,0.37,46.72,GROW,1.06
mudayrah-ajman,Ajman,18651.44,18651.44,0.97,42.91,WATCH,1.00
milehah-sharjah,Sharjah,18518.19,18508.74,0.52,43.26,GROW,0.99
al-helio-ajman,Ajman,18070.04,17957.61,1.00,14.62,WATCH,0.97
al-dhahir-abu-dhabi,Abu Dhabi,15987.71,15932.19,1.00,24.70,WATCH,0.86
hamidiya-ajman,Ajman,13535.53,13386.46,1.00,10.59,GROW,0.73


Sensitivity at the top area: the same site with a smaller new lounge (e.g. first-year reviews).

In [7]:
from src.market import women_15plus, affluence_weight
w = women_15plus(v3.cells, v3.emirates, 0.055); aw = w * affluence_weight(v3.cells.rent_observed, w, 0.5)
i = top.index[0]; x = AR.loc[i]; ns = frozenset(L.NOT_SCORED)
rows = []
for reviews in (rev, scored.review_count.quantile(.25), 100, 25):
    o = S.what_if_open(v3, a, aw, x.lat, x.lng, reviews, rat, not_scored=ns)
    rows.append(dict(area=i, new_lounge_reviews=round(reviews), served=round(o["served"]), net_new=round(o["net_new"]),
                     x_best=round(o["served"] / best, 2), grow_line=growth.NEW_SITE_MIN))
pd.DataFrame(rows)

,area,new_lounge_reviews,served,net_new,x_best,grow_line
0,al-dhaid-sharjah,757,57932,57932,3.11,6000
1,al-dhaid-sharjah,626,53569,53569,2.87,6000
2,al-dhaid-sharjah,100,16871,16871,0.90,6000
3,al-dhaid-sharjah,25,4951,4951,0.27,6000


## Open questions for review

1. **Top question: a typical new lounge out-pulls thin markets** (section 5). The typical new lounge (757 reviews, 4.6★) at al-dhaid would serve 57,932 women. That is 3.11× our best lounge (al-ain, 18,652) and 2.5× the best lounge site (a typical lounge at al-dhafra's site, 22,852). In 4 areas a new lounge would serve more than our best lounge does; 3 are GROW and manama-ajman is WATCH, on rent alone (question 3). At al-dhaid, the lounge p25 (626 reviews) still serves 53,569; 100 reviews serves 16,871 (0.90×); 25 reviews serves 4,951, below the 6,000 line. The salon-site reference doesn't fix this: these four areas are at the 94th–100th percentile in their emirate and the 99th–100th UAE-wide. Options:
   - give a new lounge less attractiveness (e.g. a new lounge's first-year reviews);
   - cap capacity;
   - strengthen the outside option. The stand-ins halved it, 201.5 → 113.0 (`served_demand.ipynb`).
2. **GROW areas aren't additive.** 15 of 32 GROW areas would serve more women than the area itself holds (milehah-sharjah holds 13,716 women and would serve 18,518), and 19 of 32 are within 10 km of another GROW area. Each what-if runs alone, so neighbours count the same women. Should we pick sites in sequence (open the best one, then re-run the rest)?
3. **Rent is the test that binds, and on fallback rents it is noise.** The calls are 32 GROW, 65 WATCH and 464 SKIP. Of the 65 WATCH, 48 are under 6,000 net new (3 of them also below the demand line), and 17 are big enough but fail on rent alone, with ratios from 0.50 to 0.997. No WATCH fails only on demand, and none on worker housing. GROW areas run from the 78th to the 100th percentile of their emirate's salon sites (median 90th; UAE-wide 79th–100th, median 93rd), so the `demand_pct` ≥ 0.5 line never binds for a big area. Rent does, but 10 of the 32 GROWs (all in Sharjah) sit at a ratio of exactly 1.000, because Sharjah's fallback cell rent equals its reference median. Meanwhile manama-ajman (35,094 net new) and mudayrah-ajman (18,651) miss at 0.997 for the same reason (`rent.ipynb`, question 7). 8 GROW areas (Ajman 4, Ras Al Khaimah 4) are ranked on demand UAE-wide because their emirate has under 50 reference sites.
4. **Imputed competitors carry 7 of 32 GROW areas** (data coverage under 0.5).
5. **The footprint mix.** 17 of 32 GROW areas are within 15 km of a lounge and 7 are 30+ km away. By emirate: Abu Dhabi 13, Sharjah 10, Ajman 4, Ras Al Khaimah 4, Dubai 1 (al-awir). All 5 v3 GROW areas stay GROW; 14 v3 WATCH and 9 v3 SKIP areas become GROW, plus 4 areas v3 didn't have.
6. **`NEW_SITE_MIN` = 6,000** is about the median lounge (6,046 served; a typical lounge at the median site serves 5,388). It's a first cut; with a smaller new lounge (question 1) it would need revisiting.